In [2]:
#分类树
import numpy as np
import pandas as pd
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

# 加载鸢尾花数据集
iris = load_iris()
X = iris.data
y = iris.target

# 划分训练集和测试集
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

# 使用 scikit-learn 的 CART 决策树分类器
clf = DecisionTreeClassifier()

# 训练模型
clf.fit(X_train, y_train)

# 预测测试集
y_pred = clf.predict(X_test)

# 计算准确率
accuracy = accuracy_score(y_test, y_pred)
print('准确率:', accuracy)
print('错误率:', 1 - accuracy)

# 自定义 CART 决策树分类器（简化版）
class MyDecisionTreeClassifier:
    def __init__(self, max_depth=None, min_samples_split=2):
        self.max_depth = max_depth
        self.min_samples_split = min_samples_split
        self.tree = None

    def fit(self, X, y):
        # 构建决策树
        self.tree = self._build_tree(X, y, depth=0)

    def _build_tree(self, X, y, depth):
        # 如果满足停止条件，返回叶节点
        if (self.max_depth is not None and depth >= self.max_depth) or len(X)<self.min_samples_split:
            # 返回叶节点的类别（多数类）
            unique_classes, class_counts = np.unique(y, return_counts=True)
            return unique_classes[np.argmax(class_counts)]

        # 选择最优特征和划分点
        best_feature, best_threshold = self._choose_best_feature(X, y)
        # 创建节点
        node = {'feature': best_feature, 'threshold': best_threshold}
        # 根据特征和阈值划分数据集
        left_indices = X[:, best_feature]<=best_threshold
        right_indices = X[:, best_feature]>best_threshold
        # 递归构建左右子树
        node['left'] = self._build_tree(X[left_indices], y[left_indices], depth + 1)
        node['right'] = self._build_tree(X[right_indices], y[right_indices], depth + 1)
        return node

    def _choose_best_feature(self, X, y):
        best_gini = float('inf')
        best_feature = None
        best_threshold = None
        # 遍历特征
        for feature in range(X.shape[1]):
            # 遍历特征的取值
            unique_values = np.unique(X[:, feature])
            for value in unique_values[:-1]:
                # 划分数据集
                left_indices = X[:, feature]<=value
                right_indices = X[:, feature]>value
                # 计算基尼指数
                gini = self._gini_index(y[left_indices], y[right_indices])
                if gini<best_gini:
                    best_gini = gini
                    best_feature = feature
                    best_threshold = value
        return best_feature, best_threshold

    def _gini_index(self, y_left, y_right):
        # 计算左右子节点的基尼指数
        def gini(y):
            unique_classes, class_counts = np.unique(y, return_counts=True)
            gini = 1
            for count in class_counts:
                gini -= (count/len(y))**2
            return gini
        # 计算总的基尼指数
        total_samples = len(y_left)+len(y_right)
        return (len(y_left)/total_samples)*gini(y_left)+(len(y_right)/total_samples)*gini(y_right)

    def predict(self, X):
        # 预测数据
        y_pred = []
        for x in X:
            node = self.tree
            while isinstance(node, dict):
                if x[node['feature']]<=node['threshold']:
                    node = node['left']
                else:
                    node = node['right']
            y_pred.append(node)
        return np.array(y_pred)

# 使用自定义 CART 决策树分类器
# my_clf = MyDecisionTreeClassifier(max_depth=5)
# my_clf.fit(X_train, y_train)
# my_y_pred = my_clf.predict(X_test)
# my_accuracy = accuracy_score(y_test, my_y_pred)
# print('自定义分类器准确率:', my_accuracy)


准确率: 1.0
错误率: 0.0


In [5]:
#回归树
import numpy as np
import pandas as pd
from sklearn.datasets import  fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_squared_error

# 加载波士顿房价数据集
housing = fetch_california_housing()
X = housing.data
y = housing.target

# 划分训练集和测试集
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

# 使用 scikit-learn 的 CART 决策树回归器
reg = DecisionTreeRegressor()

# 训练模型
reg.fit(X_train, y_train)

# 预测测试集
y_pred = reg.predict(X_test)

# 计算均方误差
mse = mean_squared_error(y_test, y_pred)
print('均方误差:', mse)

# 自定义 CART 决策树回归器（简化版）
class MyDecisionTreeRegressor:
    def __init__(self, max_depth=None, min_samples_split=2):
        self.max_depth = max_depth
        self.min_samples_split = min_samples_split
        self.tree = None

    def fit(self, X, y):
        # 构建决策树
        self.tree = self._build_tree(X, y, depth=0)

    def _build_tree(self, X, y, depth):
        # 如果满足停止条件，返回叶节点
        if (self.max_depth is not None and depth >= self.max_depth) or len(X)<self.min_samples_split:
            # 返回叶节点的均值
            return np.mean(y)

        # 选择最优特征和划分点
        best_feature, best_threshold = self._choose_best_feature(X, y)
        # 创建节点
        node = {'feature': best_feature, 'threshold': best_threshold}
        # 根据特征和阈值划分数据集
        left_indices = X[:, best_feature]<=best_threshold
        right_indices = X[:, best_feature]>best_threshold
        # 递归构建左右子树
        node['left'] = self._build_tree(X[left_indices], y[left_indices], depth + 1)
        node['left_size'] = len(y[left_indices])
        node['right'] = self._build_tree(X[right_indices], y[right_indices], depth + 1)
        node['right_size'] = len(y[right_indices])
        return node

    def _choose_best_feature(self, X, y):
        best_mse = float('inf')
        best_feature = None
        best_threshold = None
        # 遍历特征
        for feature in range(X.shape[1]):
            # 遍历特征的取值
            unique_values = np.unique(X[:, feature])
            for value in unique_values[:-1]:
                # 划分数据集
                left_indices = X[:, feature]<=value
                right_indices = X[:, feature] > value
                mse = self._mse_index(y[left_indices], y[right_indices])
                if mse<best_mse:
                    best_mse = mse
                    best_feature = feature
                    best_threshold = value
        return best_feature, best_threshold

    def _mse_index(self, y_left, y_right):
        # 计算左右子节点的均方误差
        def mse(y):
            return np.mean((y - np.mean(y))**2)
        # 计算总的均方误差
        total_samples = len(y_left)+len(y_right)
        return (len(y_left)/total_samples)*mse(y_left)+(len(y_right)/total_samples)*mse(y_right)

    def predict(self, X):
        # 预测数据
        y_pred = []
        for x in X:
            node = self.tree
            while isinstance(node, dict):
                if x[node['feature']]<=node['threshold']:
                    node = node['left']
                else:
                    node = node['right']
            y_pred.append(node)
        return np.array(y_pred)

# 使用自定义 CART 决策树回归器
# my_reg = MyDecisionTreeRegressor(max_depth=5)
# my_reg.fit(X_train, y_train)
# my_y_pred = my_reg.predict(X_test)
# my_mse = mean_squared_error(y_test, my_y_pred)
# print('自定义回归器均方误差:', my_mse)


均方误差: 0.5348232456858365
